# Phase Portraits and Stability Analysis of ODE Systems

This notebook provides a comprehensive visual companion to **Lessons 7–10** of the Differential Equations (MSc) course. We illustrate the key concepts through interactive phase portraits:

1. **Linear systems** — the three canonical Jordan normal form cases (real distinct eigenvalues, repeated eigenvalue, complex eigenvalues) and the trace–determinant classification diagram.
2. **Nonlinear systems** — linearisation near equilibria, nullclines, and the Hartman–Grobman theorem in action.
3. **Lyapunov functions** — visualising energy-like functions that certify stability.
4. **Periodic orbits** — limit cycles, the Poincaré–Bendixson theorem, and the Lotka–Volterra predator–prey model.

Every plot is generated from first principles using `numpy`, `scipy`, and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.figsize': (7, 7),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12,
})


def plot_linear_phase_portrait(A, title="Phase Portrait", xlim=(-3, 3), ylim=(-3, 3),
                                n_trajectories=20, T=8.0, ax=None):
    """
    Plot the phase portrait of the 2D linear system dx/dt = A x.

    Parameters
    ----------
    A : (2, 2) array – the system matrix.
    title : str – plot title.
    xlim, ylim : tuples – axis ranges.
    n_trajectories : int – number of trajectories to draw.
    T : float – integration time (forward and backward).
    ax : matplotlib Axes or None.
    """
    A = np.asarray(A, dtype=float)
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    else:
        fig = ax.figure

    # Stream plot (background vector field)
    xv = np.linspace(*xlim, 25)
    yv = np.linspace(*ylim, 25)
    X, Y = np.meshgrid(xv, yv)
    U = A[0, 0] * X + A[0, 1] * Y
    V = A[1, 0] * X + A[1, 1] * Y
    ax.streamplot(X, Y, U, V, color='lightblue', density=1.0, linewidth=0.5, arrowsize=0.8)

    # Trajectories from random initial conditions
    def rhs(t, y):
        return A @ y

    np.random.seed(42)
    angles = np.linspace(0, 2 * np.pi, n_trajectories, endpoint=False)
    r0 = 2.5
    colors = plt.cm.tab10(np.linspace(0, 1, n_trajectories))
    for i, theta in enumerate(angles):
        y0 = [r0 * np.cos(theta), r0 * np.sin(theta)]
        for direction, t_span in [("forward", (0, T)), ("backward", (0, -T))]:
            sol = solve_ivp(rhs, t_span, y0, max_step=0.05, dense_output=True)
            traj = sol.y
            mask = (traj[0] >= xlim[0]) & (traj[0] <= xlim[1]) & (traj[1] >= ylim[0]) & (traj[1] <= ylim[1])
            ax.plot(traj[0][mask], traj[1][mask], color=colors[i], linewidth=0.9, alpha=0.8)

    # Eigenvalues and eigenvectors
    eigvals, eigvecs = np.linalg.eig(A)
    info_lines = []
    for j in range(2):
        lam = eigvals[j]
        if np.isreal(lam):
            v = eigvecs[:, j].real
            # Draw eigenvector direction line
            t_line = np.linspace(-4, 4, 100)
            ax.plot(t_line * v[0], t_line * v[1], 'k--', linewidth=1.2, alpha=0.6)
            info_lines.append(f"λ{j+1} = {lam.real:.2f}")
        else:
            info_lines.append(f"λ{j+1} = {lam.real:.2f} ± {abs(lam.imag):.2f}i")

    # Mark the origin
    ax.plot(0, 0, 'ko', markersize=8, zorder=5)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f"{title}\nEigenvalues: {', '.join(info_lines)}", fontsize=13)
    ax.set_aspect('equal')
    return ax


def plot_nonlinear_phase_portrait(f, equilibria=None, title="Phase Portrait",
                                   xlim=(-3, 3), ylim=(-3, 3),
                                   initial_conditions=None, T=15.0, ax=None):
    """
    Plot the phase portrait of dx/dt = f(x) for a 2D nonlinear system.

    Parameters
    ----------
    f : callable – f(t, y) returning dy/dt as array of shape (2,).
    equilibria : list of (x, y) tuples – marked with red dots.
    title : str – plot title.
    xlim, ylim : tuples – axis ranges.
    initial_conditions : list of [x0, y0] – if None, generates automatically.
    T : float – integration time.
    ax : matplotlib Axes or None.
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    else:
        fig = ax.figure

    # Vector field
    xv = np.linspace(*xlim, 25)
    yv = np.linspace(*ylim, 25)
    X, Y = np.meshgrid(xv, yv)
    U = np.zeros_like(X)
    V = np.zeros_like(Y)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            dxy = f(0, [X[i, j], Y[i, j]])
            U[i, j] = dxy[0]
            V[i, j] = dxy[1]
    ax.streamplot(X, Y, U, V, color='lightblue', density=1.0, linewidth=0.5, arrowsize=0.8)

    # Trajectories
    if initial_conditions is None:
        np.random.seed(42)
        angles = np.linspace(0, 2 * np.pi, 16, endpoint=False)
        r0 = max(xlim[1], ylim[1]) * 0.8
        initial_conditions = [[r0 * np.cos(a), r0 * np.sin(a)] for a in angles]

    colors = plt.cm.Set2(np.linspace(0, 1, len(initial_conditions)))
    for i, y0 in enumerate(initial_conditions):
        for t_span in [(0, T), (0, -T)]:
            try:
                sol = solve_ivp(f, t_span, y0, max_step=0.05, dense_output=True)
                traj = sol.y
                mask = ((traj[0] >= xlim[0]) & (traj[0] <= xlim[1]) &
                        (traj[1] >= ylim[0]) & (traj[1] <= ylim[1]))
                ax.plot(traj[0][mask], traj[1][mask], color=colors[i], linewidth=0.9, alpha=0.8)
            except Exception:
                pass

    # Mark equilibria
    if equilibria:
        for eq in equilibria:
            ax.plot(eq[0], eq[1], 'ro', markersize=8, zorder=5, markeredgecolor='k')

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title(title, fontsize=13)
    ax.set_aspect('equal')
    return ax

print("Helper functions defined successfully.")

---

## 1. Linear Systems — Case I: Two Real Distinct Eigenvalues

A $2 \times 2$ linear system $\dot{x} = Ax$ with two **real, distinct** eigenvalues $\lambda_1 \neq \lambda_2$ can be reduced (via the eigenvector basis) to the diagonal form

$$A \sim \begin{pmatrix} \lambda_1 & 0 \\ 0 & \lambda_2 \end{pmatrix}.$$

The general solution is $x_1(t) = C_1 e^{\lambda_1 t}$, $x_2(t) = C_2 e^{\lambda_2 t}$, and by eliminating $t$ one finds power-law trajectories $x_2 \propto x_1^{\lambda_2 / \lambda_1}$.

Three qualitatively different phase portraits arise:

| Sub-case | Condition | Portrait type |
|----------|-----------|---------------|
| Both eigenvalues negative | $\lambda_1, \lambda_2 < 0$ | **Stable node** — all trajectories converge to the origin |
| Both eigenvalues positive | $\lambda_1, \lambda_2 > 0$ | **Unstable node** — all trajectories diverge from the origin |
| Opposite signs | $\lambda_1 < 0 < \lambda_2$ | **Saddle** — trajectories along one eigenvector approach, along the other they recede |

The **dashed lines** in the plots below show the eigenvector directions. Observe how trajectories become tangent to the eigenvector associated with the **smaller** eigenvalue (in absolute value) near the origin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# 1a. Stable node: both eigenvalues negative
A_stable_node = np.array([[-1, 0],
                           [0, -2]])
plot_linear_phase_portrait(A_stable_node, title="Stable Node", ax=axes[0])

# 1b. Unstable node: both eigenvalues positive
A_unstable_node = np.array([[1, 0],
                             [0, 2]])
plot_linear_phase_portrait(A_unstable_node, title="Unstable Node", ax=axes[1])

# 1c. Saddle: eigenvalues of opposite sign
# From the lecture notes: A = [[1, -1], [-4, 1]], eigenvalues 3 and -1
A_saddle = np.array([[1, -1],
                      [-4, 1]])
plot_linear_phase_portrait(A_saddle, title="Saddle Point", ax=axes[2])

plt.tight_layout()
plt.savefig('case_I_real_distinct.png', dpi=150, bbox_inches='tight')
plt.show()

print("Case I — Real distinct eigenvalues:")
for name, A in [("Stable node", A_stable_node), ("Unstable node", A_unstable_node), ("Saddle", A_saddle)]:
    eigvals = np.linalg.eigvals(A)
    T, D = np.trace(A), np.linalg.det(A)
    print(f"  {name}: λ = {eigvals}, Tr = {T:.1f}, Det = {D:.1f}")

### Observations — Case I

**Stable node** ($\lambda_1 = -1, \lambda_2 = -2$): All trajectories converge to the origin. Near the origin, they become tangent to the eigenvector of the **slower** eigenvalue ($\lambda_1 = -1$, along the $x_1$-axis), because the component $C_2 e^{-2t}$ decays faster than $C_1 e^{-t}$.

**Unstable node** ($\lambda_1 = 1, \lambda_2 = 2$): The time-reversed picture of the stable node. All trajectories diverge from the origin. Near the origin they are tangent to the **slower-growing** direction ($\lambda_1 = 1$).

**Saddle** ($\lambda_1 = 3, \lambda_2 = -1$): The stable manifold (trajectories approaching the origin) aligns with the eigenvector for $\lambda_2 = -1$, namely $v_2 = (1, 2)^\top$. The unstable manifold aligns with $v_1 = (1, -2)^\top$. All other trajectories first approach along the stable direction, then veer off along the unstable one — the characteristic "hyperbolic" pattern.

---

## 2. Linear Systems — Case II: Repeated Eigenvalue (Defective Matrix)

When $A$ has a **double eigenvalue** $\lambda$ but only **one linearly independent eigenvector** (i.e., $A$ is not diagonalisable), the Jordan normal form is

$$A \sim \begin{pmatrix} \lambda & 1 \\ 0 & \lambda \end{pmatrix}.$$

The general solution involves a $te^{\lambda t}$ term from the generalised eigenvector:
$$x(t) = C_1 e^{\lambda t} v_1 + C_2 e^{\lambda t}(t v_1 + v_2),$$
where $(A - \lambda I)v_2 = v_1$. This gives **degenerate (improper) nodes**: all trajectories approach (or leave) the origin tangent to the single eigenvector direction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# 2a. Stable degenerate node (λ < 0)
A_stable_degen = np.array([[-3, 1],
                            [0, -3]])
plot_linear_phase_portrait(A_stable_degen, title="Stable Degenerate Node (λ = −3)", ax=axes[0])

# 2b. Unstable degenerate node (λ > 0)
# From the lecture: A = [[2, 1], [-1, 4]], double eigenvalue λ = 3
A_unstable_degen = np.array([[2, 1],
                              [-1, 4]])
plot_linear_phase_portrait(A_unstable_degen, title="Unstable Degenerate Node (λ = 3)", ax=axes[1])

plt.tight_layout()
plt.savefig('case_II_degenerate.png', dpi=150, bbox_inches='tight')
plt.show()

for name, A in [("Stable degen.", A_stable_degen), ("Unstable degen.", A_unstable_degen)]:
    eigvals = np.linalg.eigvals(A)
    print(f"  {name}: eigenvalues = {eigvals}, Tr = {np.trace(A):.1f}, Det = {np.linalg.det(A):.1f}")

### Observations — Case II

In the **stable degenerate node**, all trajectories converge to the origin and become tangent to the single eigenvector direction as $t \to +\infty$. The $te^{\lambda t}$ factor causes trajectories to curve before settling into the asymptotic direction — unlike a proper node where different trajectories approach along different eigenvector directions.

In the **unstable degenerate node** (from the lecture notes, $A = \begin{pmatrix} 2 & 1 \\ -1 & 4 \end{pmatrix}$ with $\lambda = 3$), trajectories diverge from the origin, all tangent to $v = (1, 1)^\top$ as $t \to -\infty$.

Notice the characteristic "one-sided" curvature that distinguishes degenerate nodes from proper nodes.

---

## 3. Linear Systems — Case III: Complex Eigenvalues $\alpha \pm \beta i$

When the eigenvalues are complex conjugate, the Jordan normal form is

$$A \sim \begin{pmatrix} \alpha & -\beta \\ \beta & \alpha \end{pmatrix}.$$

In polar coordinates $(r, \varphi)$, this becomes $\dot{r} = \alpha r$, $\dot{\varphi} = \beta$, giving

$$r(t) = r_0 e^{\alpha t}, \qquad \varphi(t) = \beta t + \varphi_0.$$

- $\alpha < 0$: **Stable focus** — trajectories spiral inward.
- $\alpha > 0$: **Unstable focus** — trajectories spiral outward.
- $\alpha = 0$: **Centre** — trajectories are closed ellipses (no radial motion, only rotation).

The rotation direction depends on the sign of $\beta$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# 3a. Stable focus (α < 0)
# From lecture: A = [[ω, -1], [1, ω]] with ω < 0
omega = -0.3
A_stable_focus = np.array([[omega, -1],
                            [1, omega]])
plot_linear_phase_portrait(A_stable_focus, title=f"Stable Focus (ω = {omega})", ax=axes[0], T=15)

# 3b. Unstable focus (α > 0)
omega = 0.3
A_unstable_focus = np.array([[omega, -1],
                              [1, omega]])
plot_linear_phase_portrait(A_unstable_focus, title=f"Unstable Focus (ω = {omega})", ax=axes[1], T=15)

# 3c. Centre (α = 0)
# From lecture: A = [[0, 1], [-1, 0]] — rotation matrix
A_centre = np.array([[0, 1],
                      [-1, 0]])
plot_linear_phase_portrait(A_centre, title="Centre (ω = 0)", ax=axes[2], T=10)

plt.tight_layout()
plt.savefig('case_III_complex.png', dpi=150, bbox_inches='tight')
plt.show()

for name, A in [("Stable focus", A_stable_focus), ("Unstable focus", A_unstable_focus), ("Centre", A_centre)]:
    eigvals = np.linalg.eigvals(A)
    print(f"  {name}: eigenvalues = {eigvals[0]:.3f}, Tr = {np.trace(A):.2f}, Det = {np.linalg.det(A):.2f}")

### Observations — Case III

**Stable focus** ($\omega = -0.3$): The negative real part $\alpha = \omega$ causes exponential decay of the radius $r(t) = r_0 e^{\omega t}$, while $\dot{\varphi} = 1$ drives constant-speed rotation. The result is an inward spiral.

**Unstable focus** ($\omega = 0.3$): The time-reversed version — trajectories spiral outward.

**Centre** ($\omega = 0$): With $\alpha = 0$, the radius is constant ($r(t) = r_0$), and trajectories are perfect circles (or ellipses in the original non-diagonal coordinates). This is the **structurally unstable** case: any small perturbation to $\alpha$ turns the centre into a focus. That is why the Hartman–Grobman theorem excludes purely imaginary eigenvalues.

---

## 4. The Trace–Determinant Classification Diagram

The type of a $2 \times 2$ linear system is completely determined by the **trace** $T = \mathrm{tr}\,A$ and **determinant** $D = \det A$ of the matrix. The characteristic equation is $\lambda^2 - T\lambda + D = 0$, and the discriminant $\Delta = T^2 - 4D$ separates regions:

- $D < 0$: saddle (eigenvalues have opposite signs).
- $D > 0$, $\Delta > 0$: node (stable if $T < 0$, unstable if $T > 0$).
- $D > 0$, $\Delta = 0$: degenerate node (or star node).
- $D > 0$, $\Delta < 0$: focus (stable if $T < 0$, unstable if $T > 0$) or centre ($T = 0$).

The parabola $T^2 = 4D$ separates nodes from foci.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Draw the (T, D) diagram
T_range = np.linspace(-8, 8, 500)

# Parabola T^2 = 4D  =>  D = T^2/4
D_parabola = T_range**2 / 4
ax.plot(T_range, D_parabola, 'k-', linewidth=2, label='$T^2 = 4D$ (parabola)')

# D = 0 line
ax.axhline(0, color='k', linewidth=1.5, linestyle='-')
# T = 0 line
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')

# Fill regions with labels
# Saddle: D < 0
ax.fill_between(T_range, -5, 0, alpha=0.15, color='red')
ax.text(0, -2.5, 'SADDLE\n($D < 0$)', ha='center', va='center', fontsize=13, fontweight='bold', color='darkred')

# Stable node: D > 0, T < 0, T^2 > 4D
ax.fill_between(T_range[T_range < 0], np.maximum(0, D_parabola[T_range < 0]),
                np.full(np.sum(T_range < 0), 20), alpha=0.15, color='blue', zorder=0)
ax.text(-5.5, 12, 'STABLE\nNODE', ha='center', va='center', fontsize=12, fontweight='bold', color='darkblue')

# Unstable node: D > 0, T > 0, T^2 > 4D
ax.fill_between(T_range[T_range > 0], np.maximum(0, D_parabola[T_range > 0]),
                np.full(np.sum(T_range > 0), 20), alpha=0.15, color='orange', zorder=0)
ax.text(5.5, 12, 'UNSTABLE\nNODE', ha='center', va='center', fontsize=12, fontweight='bold', color='darkorange')

# Stable focus: D > T^2/4, T < 0
ax.text(-2.0, 6, 'STABLE\nFOCUS', ha='center', va='center', fontsize=12, fontweight='bold', color='navy')

# Unstable focus: D > T^2/4, T > 0
ax.text(2.0, 6, 'UNSTABLE\nFOCUS', ha='center', va='center', fontsize=12, fontweight='bold', color='brown')

# Centre on D-axis
ax.text(0.6, 3.5, 'CENTRE\n($T=0$)', ha='left', va='center', fontsize=11, fontstyle='italic', color='green')

# Plot our examples from the three cases
examples = [
    (np.trace(A_stable_node), np.linalg.det(A_stable_node), 'Stable node', 's', 'blue'),
    (np.trace(A_unstable_node), np.linalg.det(A_unstable_node), 'Unstable node', 's', 'orange'),
    (np.trace(A_saddle), np.linalg.det(A_saddle), 'Saddle', 's', 'red'),
    (np.trace(A_stable_degen), np.linalg.det(A_stable_degen), 'Stable degen.', 'D', 'blue'),
    (np.trace(A_unstable_degen), np.linalg.det(A_unstable_degen), 'Unstable degen.', 'D', 'orange'),
    (np.trace(A_stable_focus), np.linalg.det(A_stable_focus), 'Stable focus', 'o', 'navy'),
    (np.trace(A_unstable_focus), np.linalg.det(A_unstable_focus), 'Unstable focus', 'o', 'brown'),
    (np.trace(A_centre), np.linalg.det(A_centre), 'Centre', '*', 'green'),
]

for T, D, label, marker, color in examples:
    ax.plot(T, D, marker=marker, color=color, markersize=12, markeredgecolor='k',
            markeredgewidth=1, zorder=10, label=label)

ax.set_xlim(-8, 8)
ax.set_ylim(-5, 20)
ax.set_xlabel('Trace $T = \\mathrm{tr}\\,A$', fontsize=14)
ax.set_ylabel('Determinant $D = \\det A$', fontsize=14)
ax.set_title('Classification of 2D Linear Systems\nin the Trace–Determinant Plane', fontsize=15)
ax.legend(loc='upper left', fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trace_det_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

### Reading the Trace–Determinant Diagram

The coloured markers show where each of our previous examples sits in the $(T, D)$ plane. Notice:

- All saddle points lie below the $D = 0$ axis (red region).
- The parabola $T^2 = 4D$ cleanly separates nodes (outside) from foci (inside).
- Degenerate nodes sit exactly on the parabola.
- The centre sits on the $D$-axis ($T = 0$).

This diagram is a powerful diagnostic: given any $2 \times 2$ matrix, compute $T$ and $D$, locate the point in this plane, and you immediately know the phase portrait type without computing eigenvalues explicitly.

---

## 5. Nonlinear Systems — Linearisation and the Hartman–Grobman Theorem

For a nonlinear autonomous system $\dot{x} = f(x)$, the behaviour near an equilibrium point $p$ (where $f(p) = 0$) is governed by the **Jacobian** $J = f'(p)$. The **Hartman–Grobman theorem** guarantees that if all eigenvalues of $J$ have nonzero real part (a *hyperbolic* equilibrium), the local phase portrait of the nonlinear system is topologically equivalent to that of the linearised system $\dot{y} = Jy$.

We demonstrate this with the example from the lecture notes:
$$\dot{x} = x - xy, \qquad \dot{y} = x^2 - y.$$

Equilibria: $(0, 0)$, $(1, 1)$, $(-1, 1)$.

In [ ]:
# Nonlinear system: dx/dt = x - xy,  dy/dt = x^2 - y
def f_nonlinear1(t, z):
    x, y = z
    return [x - x * y, x**2 - y]

# Equilibria and Jacobians
equilibria_1 = [(0, 0), (1, 1), (-1, 1)]

print("Equilibrium analysis:")
print("=" * 60)
for eq in equilibria_1:
    x0, y0 = eq
    # Jacobian: J = [[1 - y, -x], [2x, -1]]
    J = np.array([[1 - y0, -x0],
                   [2 * x0, -1]])
    eigvals = np.linalg.eigvals(J)
    T, D = np.trace(J), np.linalg.det(J)
    disc = T**2 - 4 * D
    print(f"\n  Equilibrium {eq}:")
    print(f"    Jacobian = {J.tolist()}")
    print(f"    Eigenvalues: {eigvals}")
    print(f"    Tr = {T:.2f}, Det = {D:.2f}, Δ = T²−4D = {disc:.2f}")
    if D < 0:
        print(f"    Classification: SADDLE")
    elif disc < 0 and T < 0:
        print(f"    Classification: STABLE FOCUS")
    elif disc < 0 and T > 0:
        print(f"    Classification: UNSTABLE FOCUS")
    elif disc < 0 and T == 0:
        print(f"    Classification: CENTRE")
    elif disc >= 0 and T < 0:
        print(f"    Classification: STABLE NODE")
    elif disc >= 0 and T > 0:
        print(f"    Classification: UNSTABLE NODE")

In [ ]:
# Phase portrait with nullclines
fig, ax = plt.subplots(figsize=(9, 9))

# Initial conditions spread around the region of interest
ics = []
for x0 in np.linspace(-2.5, 2.5, 8):
    for y0 in np.linspace(-0.5, 3.5, 8):
        ics.append([x0, y0])
# Add some near equilibria
for eq in equilibria_1:
    for dr in [0.2, -0.2]:
        ics.append([eq[0] + dr, eq[1]])
        ics.append([eq[0], eq[1] + dr])

plot_nonlinear_phase_portrait(f_nonlinear1, equilibria=equilibria_1,
                               title=r"$\dot{x} = x - xy$, $\dot{y} = x^2 - y$" + "\nwith nullclines",
                               xlim=(-2.5, 2.5), ylim=(-0.5, 3.5),
                               initial_conditions=ics, T=10, ax=ax)

# Draw nullclines
xv = np.linspace(-2.5, 2.5, 300)

# x-nullcline: x(1-y) = 0  =>  x = 0 or y = 1
ax.axvline(0, color='red', linewidth=2, linestyle='-.', alpha=0.7, label='$x$-nullcline: $x=0$')
ax.axhline(1, color='red', linewidth=2, linestyle='--', alpha=0.7, label='$x$-nullcline: $y=1$')

# y-nullcline: x^2 - y = 0  =>  y = x^2
ax.plot(xv, xv**2, color='blue', linewidth=2, linestyle='--', alpha=0.7, label='$y$-nullcline: $y=x^2$')

# Annotate equilibria
for eq, label in zip(equilibria_1, ['(0,0)\nSaddle', '(1,1)\nStable focus', '(-1,1)\nStable focus']):
    ax.annotate(label, xy=eq, xytext=(eq[0]+0.3, eq[1]+0.35),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))

ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.savefig('nonlinear_example1.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations — Nonlinear System

The **nullclines** partition the phase plane into regions where $\dot{x}$ and $\dot{y}$ have definite signs. Equilibria are at the intersections of the $x$-nullcline ($x = 0$ or $y = 1$) and the $y$-nullcline ($y = x^2$).

- **(0, 0)** is a **saddle**: the Jacobian has eigenvalues $+1$ and $-1$. Trajectories approach along the $y$-axis and escape along the $x$-axis.
- **(1, 1)** and **(-1, 1)** are **stable foci**: eigenvalues have negative real part with nonzero imaginary part ($T = -1$, $D = 2$, $\Delta = -7$). Trajectories spiral inward.

The Hartman–Grobman theorem guarantees that the local behaviour near each hyperbolic equilibrium matches the linear prediction. The global picture — which basin of attraction each trajectory falls into — requires the nonlinear analysis with nullclines.

---

## 6. Lyapunov Functions — Visualising Stability

A **Lyapunov function** $V(x, y)$ is a positive-definite, continuously differentiable function whose **Lie derivative** $L_f V = \nabla V \cdot f$ tells us how $V$ changes along solutions:

- $L_f V \leq 0$ everywhere: the equilibrium is **stable** (trajectories cannot escape level sets of $V$).
- $L_f V < 0$ everywhere (except at the equilibrium): **asymptotically stable** (trajectories are drawn inward).
- $L_f V \equiv 0$: $V$ is a **first integral** — level curves are trajectories themselves.

We illustrate with the system $\dot{x} = -y - x^3$, $\dot{y} = x - y^3$ from the lecture, using $V(x, y) = x^2 + y^2$.

In [ ]:
# Lyapunov function demonstration
# System: dx/dt = -y - x^3,  dy/dt = x - y^3
def f_lyapunov(t, z):
    x, y = z
    return [-y - x**3, x - y**3]

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))

# Left panel: phase portrait with V contours
ax = axes[0]
xv = np.linspace(-2, 2, 30)
yv = np.linspace(-2, 2, 30)
X, Y = np.meshgrid(xv, yv)

# V(x,y) = x^2 + y^2 contours
V = X**2 + Y**2
ax.contour(X, Y, V, levels=np.linspace(0.2, 4, 10), colors='green', alpha=0.5, linewidths=1.5)

# Trajectories
ics = [[r * np.cos(a), r * np.sin(a)] for r in [0.5, 1.0, 1.5, 2.0]
       for a in np.linspace(0, 2 * np.pi, 8, endpoint=False)]
colors_traj = plt.cm.plasma(np.linspace(0.1, 0.9, len(ics)))
for i, y0 in enumerate(ics):
    sol = solve_ivp(f_lyapunov, (0, 15), y0, max_step=0.02)
    ax.plot(sol.y[0], sol.y[1], color=colors_traj[i], linewidth=0.8, alpha=0.9)

ax.plot(0, 0, 'ko', markersize=8, zorder=5)
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Phase portrait with $V = x^2 + y^2$ contours (green)', fontsize=12)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right panel: V(t) along several trajectories — show it decreasing
ax2 = axes[1]
for r0 in [0.5, 1.0, 1.5, 2.0]:
    y0 = [r0, 0]
    sol = solve_ivp(f_lyapunov, (0, 10), y0, max_step=0.02, dense_output=True)
    t_eval = np.linspace(0, 10, 500)
    traj = sol.sol(t_eval)
    V_t = traj[0]**2 + traj[1]**2
    ax2.plot(t_eval, V_t, linewidth=2, label=f'$r_0 = {r0}$')

ax2.set_xlabel('$t$', fontsize=13)
ax2.set_ylabel('$V(x(t), y(t)) = x^2 + y^2$', fontsize=13)
ax2.set_title('$V$ is strictly decreasing along trajectories\n'
              r'$L_f V = -2(x^4 + y^4) < 0$ $\Rightarrow$ asympt. stable', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lyapunov_demo.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations — Lyapunov Function

**Left panel:** The green circles are level curves of $V(x,y) = x^2 + y^2$. Every trajectory crosses these circles **inward** — the "energy" $V$ is always decreasing. The trajectories spiral into the origin, confirming asymptotic stability.

**Right panel:** The function $V(x(t), y(t))$ plotted as a function of time for four different starting radii. All curves are strictly decreasing. This is exactly the statement that $L_f V = -2(x^4 + y^4) < 0$ for $(x,y) \neq (0,0)$.

Note that the Jacobian at the origin is $\begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ with eigenvalues $\pm i$ — the linearisation gives a **centre**, which tells us nothing about stability! The Lyapunov function reveals that the nonlinear terms ($-x^3$ and $-y^3$) provide the damping that makes the origin asymptotically stable.

---

## 7. First Integral — Global Phase Portrait via $L_f V \equiv 0$

When the Lie derivative vanishes identically ($L_f V = 0$), the function $V$ is a **first integral**: it is constant along every trajectory. The level curves $V = c$ **are** the trajectories. We demonstrate with:

$$\dot{x} = -xy^4, \qquad \dot{y} = x^6 y, \qquad V(x,y) = \frac{x^6}{6} + \frac{y^4}{4}.$$

In [ ]:
# First integral example: dx/dt = -xy^4, dy/dt = x^6 y
# V(x,y) = x^6/6 + y^4/4   =>  L_f V = 0  (verify: x^5(-xy^4) + y^3(x^6 y) = -x^6 y^4 + x^6 y^4 = 0)

fig, ax = plt.subplots(figsize=(8, 8))

xv = np.linspace(-1.8, 1.8, 400)
yv = np.linspace(-1.8, 1.8, 400)
X, Y = np.meshgrid(xv, yv)

V = X**6 / 6 + Y**4 / 4

# Level curves of V are the trajectories
levels = np.linspace(0.01, 1.5, 20)
cs = ax.contour(X, Y, V, levels=levels, cmap='viridis', linewidths=1.5)
ax.clabel(cs, fontsize=8, fmt='%.2f')

# Add direction arrows using the vector field
U = -X * Y**4
W = X**6 * Y
# Normalise for quiver
speed = np.sqrt(U**2 + W**2)
speed[speed == 0] = 1
U_norm = U / speed
W_norm = W / speed
# Subsample for quiver
step = 20
ax.quiver(X[::step, ::step], Y[::step, ::step],
          U_norm[::step, ::step], W_norm[::step, ::step],
          color='gray', alpha=0.5, scale=25)

ax.plot(0, 0, 'ko', markersize=8, zorder=5)
ax.set_xlabel('$x$', fontsize=13)
ax.set_ylabel('$y$', fontsize=13)
ax.set_title(r"First integral: $V = \frac{x^6}{6} + \frac{y^4}{4}$" + "\nLevel curves = trajectories (all solutions are periodic)",
             fontsize=12)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('first_integral.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Limit Cycles — The Poincare–Bendixson Theorem in Action

A **limit cycle** is an isolated closed trajectory. Unlike the centre (where every trajectory is closed), a limit cycle is an attractor (or repeller) — nearby trajectories spiral toward (or away from) it.

The **Poincare–Bendixson theorem** guarantees existence: if a trajectory is trapped in a compact region that contains no equilibria, it must approach a closed orbit.

We illustrate with the canonical example (in polar form: $\dot{r} = r(1 - r^2)$, $\dot{\varphi} = 1$):

$$\dot{x} = x - y - x(x^2 + y^2), \qquad \dot{y} = x + y - y(x^2 + y^2).$$

The circle $r = 1$ is a **stable limit cycle**: trajectories inside spiral outward, trajectories outside spiral inward.

In [ ]:
# Limit cycle example
def f_limit_cycle(t, z):
    x, y = z
    r2 = x**2 + y**2
    return [x - y - x * r2, x + y - y * r2]

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))

# Left: phase portrait
ax = axes[0]

# The limit cycle r = 1
theta_lc = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta_lc), np.sin(theta_lc), 'r-', linewidth=3, alpha=0.8, label='Limit cycle $r = 1$')

# Trajectories from inside (spiral outward to r=1)
for r0 in [0.1, 0.3, 0.5, 0.7]:
    for a0 in [0, np.pi/2, np.pi, 3*np.pi/2]:
        y0 = [r0 * np.cos(a0), r0 * np.sin(a0)]
        sol = solve_ivp(f_limit_cycle, (0, 25), y0, max_step=0.02)
        ax.plot(sol.y[0], sol.y[1], 'b-', linewidth=0.7, alpha=0.6)

# Trajectories from outside (spiral inward to r=1)
for r0 in [1.3, 1.6, 2.0, 2.5]:
    for a0 in [0, np.pi/2, np.pi, 3*np.pi/2]:
        y0 = [r0 * np.cos(a0), r0 * np.sin(a0)]
        sol = solve_ivp(f_limit_cycle, (0, 25), y0, max_step=0.02)
        ax.plot(sol.y[0], sol.y[1], 'g-', linewidth=0.7, alpha=0.6)

# Poincaré–Bendixson annulus
for r in [0.5, 2.0]:
    ax.plot(r * np.cos(theta_lc), r * np.sin(theta_lc), 'k:', linewidth=1.5, alpha=0.5)
ax.annotate('Trapping annulus\n$\\frac{1}{2} \\leq r \\leq 2$', xy=(1.6, 1.6),
            fontsize=11, fontstyle='italic',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.plot(0, 0, 'ko', markersize=6, zorder=5)
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_xlabel('$x$', fontsize=13)
ax.set_ylabel('$y$', fontsize=13)
ax.set_title('Stable Limit Cycle at $r = 1$\nBlue: from inside, Green: from outside', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: r(t) showing convergence to r = 1
ax2 = axes[1]
for r0, color, label in [(0.1, 'blue', '$r_0 = 0.1$'), (0.5, 'dodgerblue', '$r_0 = 0.5$'),
                          (1.5, 'green', '$r_0 = 1.5$'), (2.5, 'darkgreen', '$r_0 = 2.5$')]:
    y0 = [r0, 0]
    sol = solve_ivp(f_limit_cycle, (0, 15), y0, max_step=0.02, dense_output=True)
    t_eval = np.linspace(0, 15, 500)
    traj = sol.sol(t_eval)
    r_t = np.sqrt(traj[0]**2 + traj[1]**2)
    ax2.plot(t_eval, r_t, color=color, linewidth=2, label=label)

ax2.axhline(1, color='red', linewidth=2, linestyle='--', alpha=0.7, label='$r = 1$ (limit cycle)')
ax2.set_xlabel('$t$', fontsize=13)
ax2.set_ylabel('$r(t) = \\sqrt{x^2 + y^2}$', fontsize=13)
ax2.set_title('Radius converges to $r = 1$ from all initial conditions\n'
              '$\\dot{r} = r(1 - r^2)$: positive for $r < 1$, negative for $r > 1$', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('limit_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations — Limit Cycle

**Left panel:** The red circle $r = 1$ is the limit cycle. Blue trajectories (starting inside) spiral outward; green trajectories (starting outside) spiral inward. The dotted circles mark the **trapping annulus** $\frac{1}{2} \leq r \leq 2$: on the inner boundary $\dot{r} > 0$ (outward flow), on the outer boundary $\dot{r} < 0$ (inward flow). Since the only equilibrium (the origin) is outside the annulus, Poincare–Bendixson guarantees a closed orbit inside.

**Right panel:** $r(t)$ converges to $1$ from both sides, confirming the stability of the limit cycle. The ODE $\dot{r} = r(1 - r^2)$ has the fixed point $r = 1$ as a stable equilibrium.

---

## 9. Lotka–Volterra Predator–Prey: Closed Orbits from a First Integral

The classical Lotka–Volterra system models predator–prey dynamics:

$$\dot{x} = x - xy, \qquad \dot{y} = xy - y, \qquad x, y > 0.$$

This system has a first integral $V(x,y) = x - \ln x + y - \ln y$. Since $L_f V = 0$, the level curves of $V$ are closed orbits: the populations oscillate periodically around the equilibrium $(1, 1)$.

In [ ]:
# Lotka-Volterra predator-prey
def f_lotka_volterra(t, z):
    x, y = z
    return [x - x * y, x * y - y]

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))

# Left: phase portrait (first quadrant)
ax = axes[0]

# Level curves of the first integral V = x - ln(x) + y - ln(y)
xv = np.linspace(0.05, 5, 400)
yv = np.linspace(0.05, 5, 400)
X, Y = np.meshgrid(xv, yv)
V_lv = X - np.log(X) + Y - np.log(Y)

# V at the equilibrium (1,1) = 1 - 0 + 1 - 0 = 2
levels_lv = np.linspace(2.05, 6, 12)
cs = ax.contour(X, Y, V_lv, levels=levels_lv, cmap='coolwarm', linewidths=1.5)

# Trajectories
colors_lv = plt.cm.Set1(np.linspace(0, 0.8, 8))
for i, (x0, y0) in enumerate([(0.5, 0.5), (0.3, 1.0), (1.0, 0.3), (2.0, 2.0),
                                (3.0, 1.0), (1.0, 3.0), (0.2, 0.5), (4.0, 0.5)]):
    sol = solve_ivp(f_lotka_volterra, (0, 20), [x0, y0], max_step=0.02,
                    events=None, rtol=1e-10, atol=1e-12)
    ax.plot(sol.y[0], sol.y[1], color=colors_lv[i], linewidth=1.2, alpha=0.8)

ax.plot(1, 1, 'r*', markersize=15, zorder=5, markeredgecolor='k', label='Equilibrium (1,1)')
ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_xlabel('$x$ (prey)', fontsize=13)
ax.set_ylabel('$y$ (predator)', fontsize=13)
ax.set_title('Lotka–Volterra Phase Portrait\nClosed orbits = level curves of $V = x - \\ln x + y - \\ln y$',
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Right: x(t) and y(t) over time
ax2 = axes[1]
sol = solve_ivp(f_lotka_volterra, (0, 30), [2.0, 1.0], max_step=0.01,
                dense_output=True, rtol=1e-10, atol=1e-12)
t_eval = np.linspace(0, 30, 1000)
traj = sol.sol(t_eval)
ax2.plot(t_eval, traj[0], 'b-', linewidth=2, label='Prey $x(t)$')
ax2.plot(t_eval, traj[1], 'r-', linewidth=2, label='Predator $y(t)$')
ax2.set_xlabel('$t$', fontsize=13)
ax2.set_ylabel('Population', fontsize=13)
ax2.set_title('Population dynamics: periodic oscillations\nPredator lags behind prey (quarter-period phase shift)',
              fontsize=12)
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lotka_volterra.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations — Lotka–Volterra

**Left panel:** Every trajectory in the first quadrant is a closed orbit surrounding the equilibrium $(1, 1)$. This is because $V(x, y) = x - \ln x + y - \ln y$ is a first integral: $L_f V = 0$. The minimum of $V$ is at $(1, 1)$ where $V = 2$.

**Right panel:** Both populations oscillate periodically. The predator population (red) **lags behind** the prey (blue) by roughly a quarter period — the classic predator–prey phase relationship:
1. Abundant prey $\to$ predators increase.
2. Many predators $\to$ prey decreases.
3. Scarce prey $\to$ predators decrease.
4. Few predators $\to$ prey recovers. Repeat.

Unlike the limit cycle example, here there is no isolated periodic orbit — the entire phase portrait consists of closed orbits, one for each energy level $V = c > 2$. This is structurally unstable: adding any dissipation (e.g., logistic competition $-bx^2$) would destroy the exact periodicity and replace it with either a stable equilibrium or a limit cycle.

---

## 10. Bendixson's Criterion — Proving Non-existence of Closed Orbits

**Bendixson's criterion** states: if $\operatorname{div}(f, g) = \frac{\partial f}{\partial x} + \frac{\partial g}{\partial y}$ has **constant sign** (and is not identically zero) in a simply connected region, then **no closed orbit exists** in that region.

We visualise the divergence field for two systems from the lecture notes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Example 1: dx/dt = y, dy/dt = -x - y + x^2  (div = -1 everywhere) ---
ax = axes[0]
xv = np.linspace(-3, 3, 200)
yv = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(xv, yv)

# div(f, g) = d(y)/dx + d(-x - y + x^2)/dy = 0 + (-1) = -1
div1 = np.full_like(X, -1.0)

im1 = ax.imshow(div1, extent=(-3, 3, -3, 3), origin='lower', cmap='RdBu_r',
                vmin=-2, vmax=2, alpha=0.4, aspect='equal')
ax.contour(X, Y, div1, levels=[0], colors='k', linewidths=2)  # no zero contour exists

# Overlay stream plot
U1 = Y
V1 = -X - Y + X**2
ax.streamplot(X, Y, U1, V1, color='darkblue', density=1.5, linewidth=0.8, arrowsize=0.8)

ax.set_title('$\\dot{x} = y$, $\\dot{y} = -x - y + x^2$\n'
             '$\\mathrm{div}(f,g) = -1 < 0$ everywhere\n'
             'Bendixson: NO closed orbits possible', fontsize=11)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
plt.colorbar(im1, ax=ax, label='div$(f,g)$', shrink=0.8)

# --- Example 2: Limit cycle system — div changes sign ---
ax = axes[1]
xv = np.linspace(-2.5, 2.5, 200)
yv = np.linspace(-2.5, 2.5, 200)
X, Y = np.meshgrid(xv, yv)

# div = 2 - 4(x^2 + y^2)  for the limit cycle system
div2 = 2 - 4 * (X**2 + Y**2)

im2 = ax.imshow(div2, extent=(-2.5, 2.5, -2.5, 2.5), origin='lower', cmap='RdBu_r',
                vmin=-10, vmax=10, alpha=0.4, aspect='equal')
# Zero-divergence curve: 2 - 4r^2 = 0  =>  r = 1/sqrt(2)
theta = np.linspace(0, 2 * np.pi, 200)
r_zero = 1 / np.sqrt(2)
ax.plot(r_zero * np.cos(theta), r_zero * np.sin(theta), 'k--', linewidth=2,
        label=f'div = 0 at $r = 1/\\sqrt{{2}} \\approx {r_zero:.2f}$')

# Limit cycle at r = 1
ax.plot(np.cos(theta), np.sin(theta), 'r-', linewidth=2.5, label='Limit cycle $r=1$')

U2 = X - Y - X * (X**2 + Y**2)
V2 = X + Y - Y * (X**2 + Y**2)
ax.streamplot(X, Y, U2, V2, color='darkblue', density=1.5, linewidth=0.8, arrowsize=0.8)

ax.set_title('Limit cycle system\n'
             '$\\mathrm{div}(f,g) = 2 - 4(x^2+y^2)$: changes sign at $r = 1/\\sqrt{2}$\n'
             'Bendixson does NOT apply (div changes sign)', fontsize=11)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.legend(loc='lower right', fontsize=10)
plt.colorbar(im2, ax=ax, label='div$(f,g)$', shrink=0.8)

plt.tight_layout()
plt.savefig('bendixson.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations — Bendixson's Criterion

**Left panel:** For $\dot{x} = y$, $\dot{y} = -x - y + x^2$, the divergence is $\mathrm{div}(f,g) = 0 + (-1) = -1$ — a negative constant everywhere. Bendixson's criterion immediately rules out any closed orbit. The streamlines confirm: all trajectories converge to equilibria, with no periodic behaviour.

**Right panel:** For the limit cycle system, $\mathrm{div}(f,g) = 2 - 4(x^2 + y^2)$ changes sign at $r = 1/\sqrt{2}$. Bendixson's criterion **does not apply** because the divergence is positive inside this circle and negative outside. This is consistent with the existence of the limit cycle at $r = 1$. However, in the **annulus** $3/4 < r^2 < 5/4$, the divergence is strictly negative (between $-1$ and $-3$), which proves there is **at most one** closed orbit in that annulus — confirming the uniqueness of the limit cycle.

---

## Summary

This notebook demonstrated the complete toolkit for analysing 2D autonomous ODE systems:

| Tool | Purpose | Demonstrated in |
|------|---------|----------------|
| Jordan normal form (3 cases) | Classify linear phase portraits | Sections 1–3 |
| Trace–determinant diagram | Quick classification without eigenvalues | Section 4 |
| Jacobian + Hartman–Grobman | Local behaviour of nonlinear systems | Section 5 |
| Nullclines | Qualitative flow direction | Section 5 |
| Lyapunov functions | Stability when linearisation fails | Sections 6–7 |
| Poincare–Bendixson | Existence of limit cycles | Section 8 |
| First integrals | Exact trajectory determination | Sections 7, 9 |
| Bendixson–Dulac criteria | Non-existence of closed orbits | Section 10 |